# End-to-End Face Mask Classification Pipeline
Notebook ini mencakup:
1. SVM Tanpa Augmentasi (Canny & DWT)
2. SVM Dengan Augmentasi On-the-fly (Canny & DWT)
3. MobileNetV2 Tanpa Augmentasi
4. MobileNetV2 Dengan Augmentasi


In [ ]:
import sys
import os
# Jika script diupload sebagai dataset, hilangkan tanda pagar di bawah ini dan sesuaikan nama foldernya
sys.path.append('/kaggle/input/datasets/emageeeee/pcd-k23')

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from config import *
from src.preprocess import *
from src.feature_engineering import load_and_extract_features, scale_features
from src.train import train_svm, train_mobilenet, train_cnn
from src.evaluate import evaluate_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

MODELS_DIR.mkdir(parents=True, exist_ok=True)
kaggle_dir = download_data_from_kaggle()

# Pastikan data bersih dari gambar hitam dan duplikat
remove_blank_images(kaggle_dir)
remove_duplicate_images(kaggle_dir)

## Skenario 1: SVM Tanpa Image Enhance (Canny & DWT)

In [ ]:
print(">> Loading SVM Data (NO Enhancement)...")
X_train_c_no, X_train_d_no, y_train_no = load_and_extract_features(kaggle_dir / "Train", require_preprocess=False)
X_test_c_no, X_test_d_no, y_test_no = load_and_extract_features(kaggle_dir / "Test", require_preprocess=False)
X_train_c_s_no, _, X_test_c_s_no, _ = scale_features(X_train_c_no, None, X_test_c_no)
X_train_d_s_no, _, X_test_d_s_no, _ = scale_features(X_train_d_no, None, X_test_d_no)
svm_c_no = train_svm(X_train_c_s_no, y_train_no, MODELS_DIR / 'svm_c_no_enh.pkl')
evaluate_model(y_test_no, svm_c_no.predict(X_test_c_s_no), CLASSES, "SVM Canny (Tanpa Enhancement)")
svm_d_no = train_svm(X_train_d_s_no, y_train_no, MODELS_DIR / 'svm_d_no_enh.pkl')
evaluate_model(y_test_no, svm_d_no.predict(X_test_d_s_no), CLASSES, "SVM DWT (Tanpa Enhancement)")

## Skenario 2: SVM Dengan Image Enhance (Canny & DWT)

In [ ]:
print("\n>> Loading SVM Data (WITH Enhancement)...")
X_train_c_enh, X_train_d_enh, y_train_enh = load_and_extract_features(kaggle_dir / "Train", require_preprocess=True)
X_test_c_enh, X_test_d_enh, y_test_enh = load_and_extract_features(kaggle_dir / "Test", require_preprocess=True)
X_train_c_s_enh, _, X_test_c_s_enh, _ = scale_features(X_train_c_enh, None, X_test_c_enh)
X_train_d_s_enh, _, X_test_d_s_enh, _ = scale_features(X_train_d_enh, None, X_test_d_enh)
svm_c_enh = train_svm(X_train_c_s_enh, y_train_enh, MODELS_DIR / 'svm_c_enh.pkl')
evaluate_model(y_test_enh, svm_c_enh.predict(X_test_c_s_enh), CLASSES, "SVM Canny (Dengan Enhancement)")
svm_d_enh = train_svm(X_train_d_s_enh, y_train_enh, MODELS_DIR / 'svm_d_enh.pkl')
evaluate_model(y_test_enh, svm_d_enh.predict(X_test_d_s_enh), CLASSES, "SVM DWT (Dengan Enhancement)")

## Setup Generator CNN

In [ ]:
datagen_cnn_no_enh = ImageDataGenerator(rescale=1./255, preprocessing_function=cnn_prep_no_enhance)
datagen_cnn_enh = ImageDataGenerator(rescale=1./255, preprocessing_function=cnn_prep_enhance)

## Dengan Image Enhancement

In [ ]:
print("\n--- Training Custom CNN (TANPA Enhancement) ---")
train_gen_cnn_no = datagen_cnn_no_enh.flow_from_directory(kaggle_dir/"Train", target_size=IMG_SIZE_CNN, batch_size=BATCH_SIZE, class_mode='categorical')
val_gen_cnn_no = datagen_cnn_no_enh.flow_from_directory(kaggle_dir/"Validation", target_size=IMG_SIZE_CNN, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)
test_gen_cnn_no = datagen_cnn_no_enh.flow_from_directory(kaggle_dir/"Test", target_size=IMG_SIZE_CNN, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)
cnn_no, _ = train_cnn(train_gen_cnn_no, val_gen_cnn_no, MODELS_DIR / 'cnn_no_enh.h5', epochs=10)
evaluate_model(test_gen_cnn_no.classes, np.argmax(cnn_no.predict(test_gen_cnn_no), axis=1), CLASSES, "Custom CNN (Tanpa Enhancement)")

## Tanpa Image Enhancement

In [ ]:
print("\n--- Training Custom CNN (DENGAN Enhancement) ---")
train_gen_cnn_enh = datagen_cnn_enh.flow_from_directory(kaggle_dir/"Train", target_size=IMG_SIZE_CNN, batch_size=BATCH_SIZE, class_mode='categorical')
val_gen_cnn_enh = datagen_cnn_enh.flow_from_directory(kaggle_dir/"Validation", target_size=IMG_SIZE_CNN, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)
test_gen_cnn_enh = datagen_cnn_enh.flow_from_directory(kaggle_dir/"Test", target_size=IMG_SIZE_CNN, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)
cnn_enh, _ = train_cnn(train_gen_cnn_enh, val_gen_cnn_enh, MODELS_DIR / 'cnn_enh.h5', epochs=10)
evaluate_model(test_gen_cnn_enh.classes, np.argmax(cnn_enh.predict(test_gen_cnn_enh), axis=1), CLASSES, "Custom CNN (Dengan Enhancement)")

## Setup Generator MobileNetV2

In [ ]:
datagen_mob_no_enh = ImageDataGenerator(preprocessing_function=mobilenet_prep_no_enhance)
datagen_mob_enh = ImageDataGenerator(preprocessing_function=mobilenet_prep_enhance)

## Skenario 3: MobileNetV2 Tanpa Image Enhance

In [ ]:
print("\n--- Training MobileNetV2 (TANPA Enhancement) ---")
train_gen_mob_no = datagen_mob_no_enh.flow_from_directory(kaggle_dir/"Train", target_size=IMG_SIZE_CNN, batch_size=BATCH_SIZE, class_mode='categorical')
val_gen_mob_no = datagen_mob_no_enh.flow_from_directory(kaggle_dir/"Validation", target_size=IMG_SIZE_CNN, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)
test_gen_mob_no = datagen_mob_no_enh.flow_from_directory(kaggle_dir/"Test", target_size=IMG_SIZE_CNN, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)
mob_no, _ = train_mobilenet(train_gen_mob_no, val_gen_mob_no, MODELS_DIR / 'mob_no_enh.h5', epochs=5)
evaluate_model(test_gen_mob_no.classes, np.argmax(mob_no.predict(test_gen_mob_no), axis=1), CLASSES, "MobileNetV2 (Tanpa Enhancement)")

## Skenario 4: MobileNetV2 Dengan Image Enhance

In [ ]:
print("\n--- Training MobileNetV2 (DENGAN Enhancement) ---")
train_gen_mob_enh = datagen_mob_enh.flow_from_directory(kaggle_dir/"Train", target_size=IMG_SIZE_CNN, batch_size=BATCH_SIZE, class_mode='categorical')
val_gen_mob_enh = datagen_mob_enh.flow_from_directory(kaggle_dir/"Validation", target_size=IMG_SIZE_CNN, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)
test_gen_mob_enh = datagen_mob_enh.flow_from_directory(kaggle_dir/"Test", target_size=IMG_SIZE_CNN, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)
mob_enh, _ = train_mobilenet(train_gen_mob_enh, val_gen_mob_enh, MODELS_DIR / 'mob_enh.h5', epochs=5)
evaluate_model(test_gen_mob_enh.classes, np.argmax(mob_enh.predict(test_gen_mob_enh), axis=1), CLASSES, "MobileNetV2 (Dengan Enhancement)")